# ML-08 — Capstone Modeling Lane

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/flyrank-bih/flyrank-ml-internship-starter/blob/main/work/notebooks/w05_model.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

In [4]:
import os
import duckdb
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import precision_score, recall_score, roc_auc_score, confusion_matrix
from google.colab import userdata

# 1. Retrieve Secret Token
try:
    HF_TOKEN = userdata.get("HF_TOKEN")
    os.environ["HF_TOKEN"] = HF_TOKEN
except Exception as e:
    raise RuntimeError("Ensure 'HF_TOKEN' is set in Google Colab Secrets.") from e

# 2. Setup DuckDB Connection with Hugging Face Secret
con = duckdb.connect()
con.execute(f"CREATE OR REPLACE SECRET hf (TYPE HUGGINGFACE, TOKEN '{HF_TOKEN}');")

REL = 'hf://datasets/FlyRank/internship-warehouse'
TABLES = {
    'dim_clients': f"read_parquet('{REL}/dim_clients.parquet')",
    'dim_content': f"read_parquet('{REL}/dim_content.parquet')",
    'fact_daily':  f"read_parquet('{REL}/fact_content_daily_performance/**/*.parquet')",
}

print("DuckDB connected and Hugging Face secret configured.")

DuckDB connected and Hugging Face secret configured.


In [5]:
# Extract panel: March 2026 features predicting April 2026 refresh signals
# (June 2026 remains completely sealed for final evaluation)

query = f"""
WITH march_features AS (
    SELECT
        content_hash_id,
        client_hash_id,
        SUM(gsc_impressions) AS impressions,
        SUM(gsc_clicks) AS clicks,
        AVG(gsc_avg_position) AS avg_position,
        SUM(ga4_sessions) AS sessions,
        CASE
            WHEN SUM(gsc_impressions) = 0 THEN 0.0
            ELSE 100.0 * SUM(gsc_clicks) / SUM(gsc_impressions)
        END AS ctr
    FROM {TABLES['fact_daily']}
    WHERE month = '2026-03'
      AND gsc_data_available IS TRUE
      AND ga4_data_available IS TRUE
    GROUP BY content_hash_id, client_hash_id
),
april_performance AS (
    SELECT
        content_hash_id,
        -- Define Target Label: High search visibility but underperforming outcome in subsequent month
        CASE
            WHEN SUM(gsc_impressions) >= 1000
             AND (100.0 * SUM(gsc_clicks) / NULLIF(SUM(gsc_impressions), 0)) < 1.0
            THEN 1
            ELSE 0
        END AS target_label
    FROM {TABLES['fact_daily']}
    WHERE month = '2026-04'
      AND gsc_data_available IS TRUE
    GROUP BY content_hash_id
)
SELECT
    f.content_hash_id,
    f.client_hash_id,
    f.impressions,
    f.clicks,
    f.avg_position,
    f.sessions,
    f.ctr,
    COALESCE(p.target_label, 0) AS target
FROM march_features f
LEFT JOIN april_performance p ON f.content_hash_id = p.content_hash_id;
"""

print("Executing SQL query for March -> April temporal panel...")
df_panel = con.sql(query).df()
print(f"Dataset shape: {df_panel.shape}")
print(f"Target distribution:\n{df_panel['target'].value_counts(normalize=True)}")

Executing SQL query for March -> April temporal panel...


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Dataset shape: (63856, 8)
Target distribution:
target
0    0.553668
1    0.446332
Name: proportion, dtype: float64


## 1. Method choice and why

*Which method from the toolkit, and why it fits your lane.*

- **Task**: Priority ranking of content pages that need a refresh.
- **Approach**: Binary classification outputting probability scores ($P(\text{needs\_refresh})$) to rank pages continuously.
- **Selected Method**: **Random Forest Classifier** benchmarked against **Logistic Regression**.
- **Justification**:
  1. Non-linear dependencies exist between impressions, ranking position, and CTR decay.
  2. Decision trees naturally handle unscaled features and skew without artificial normalizations.
  3. Feature importance scores offer clear interpretability to verify signal validity against domain logic.
"""

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## 2. Split design

*Grouped by client? Time-aware? Say why this split is honest for your question.*

## 2. Split Design & Rationale

### Split Strategy
* **Time-Aware Temporal Split ($t \rightarrow t+1$):** Input features are extracted strictly from **March 2026** performance metrics, while target labels are constructed from **April 2026** outcome signals.
* **Sealed Holdout Window:** The final panel month (**June 2026**, represented by `fact_content_daily_performance_sample`) remains completely sealed as an isolated test set. It is never accessed during feature creation, model fitting, or validation.
* **Client Panel Filtering:** Filtering requires active performance flags (`gsc_data_available = TRUE` and `ga4_data_available = TRUE`), ensuring that clients with insufficient history (per `dim_clients.gsc_data_start`) do not inject false zero-filled noise into the dataset.
* **Validation Strategy:** A stratified 80/20 train/validation split is applied to the March $\rightarrow$ April panel to preserve target class proportions across subsets.

---

### Why This Split Is Honest
1. **Zero Temporal Leakage:** Feature matrices utilize exclusively historical data available at the decision boundary. No future window variables or post-event aggregations contaminate the feature set.
2. **Realistic Evaluation:** Evaluating on an unseen 20% validation split mirrors production deployment, ensuring metrics (Precision@20, Precision@50, and ROC-AUC) reflect true generalization rather than memorization.
3. **No Target Contamination:** Training on mid-panel months (March $\rightarrow$ April) prevents optimizing hyperparameters directly inside the outcome window of the sealed final month.

In [6]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
from sklearn.model_selection import train_test_split

# 1. Define Features & Target
feature_cols = ['impressions', 'clicks', 'avg_position', 'sessions', 'ctr']
X = df_panel[feature_cols].fillna(0)
y = df_panel['target']

# 2. Stratified Train / Validation Split
X_train, X_val, y_train, y_val = train_test_split(
    X, y, test_size=0.20, random_state=42, stratify=y
)

## 3. Train + compare vs my baseline

*Same data, same metric, same split as your Week-4 baseline. Show the table.*

In [7]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
# 3. Baseline Rule Scoring (Week 4 ML-07 Baseline)
def baseline_rule_score(df_in):
    scores = np.zeros(len(df_in))
    scores[(df_in['impressions'] >= 1000)] += 40
    scores[(df_in['ctr'] < 1.0)] += 30
    scores[(df_in['avg_position'] > 10)] += 30
    return scores

val_baseline_scores = baseline_rule_score(X_val)

# 4. Train Learned Models
# Logistic Regression
log_reg = LogisticRegression(random_state=42, max_iter=1000)
log_reg.fit(X_train, y_train)
lr_probs = log_reg.predict_proba(X_val)[:, 1]

# Random Forest Classifier
rf_model = RandomForestClassifier(n_estimators=100, max_depth=6, random_state=42, n_jobs=-1)
rf_model.fit(X_train, y_train)
rf_probs = rf_model.predict_proba(X_val)[:, 1]

# 5. Helper function for Top-K Precision Evaluation
def precision_at_k(y_true, scores, k=20):
    top_indices = np.argsort(scores)[::-1][:k]
    return np.mean(y_true.iloc[top_indices])

# 6. Build Comparison Metrics Table
results = [
    {
        "Model / Method": "Base Rate",
        "Precision@20": f"{y_val.mean():.4f}",
        "Precision@50": f"{y_val.mean():.4f}",
        "ROC-AUC": "0.5000"
    },
    {
        "Model / Method": "Week-4 Rule Baseline",
        "Precision@20": f"{precision_at_k(y_val, val_baseline_scores, 20):.4f}",
        "Precision@50": f"{precision_at_k(y_val, val_baseline_scores, 50):.4f}",
        "ROC-AUC": f"{roc_auc_score(y_val, val_baseline_scores):.4f}"
    },
    {
        "Model / Method": "Logistic Regression",
        "Precision@20": f"{precision_at_k(y_val, lr_probs, 20):.4f}",
        "Precision@50": f"{precision_at_k(y_val, lr_probs, 50):.4f}",
        "ROC-AUC": f"{roc_auc_score(y_val, lr_probs):.4f}"
    },
    {
        "Model / Method": "Random Forest (max_depth=6)",
        "Precision@20": f"{precision_at_k(y_val, rf_probs, 20):.4f}",
        "Precision@50": f"{precision_at_k(y_val, rf_probs, 50):.4f}",
        "ROC-AUC": f"{roc_auc_score(y_val, rf_probs):.4f}"
    }
]

df_comparison = pd.DataFrame(results)
print("=== MODEL VS BASELINE COMPARISON TABLE ===")
df_comparison

=== MODEL VS BASELINE COMPARISON TABLE ===


,Model / Method,Precision@20,Precision@50,ROC-AUC
0,Base Rate,0.4464,0.4464,0.5000
1,Week-4 Rule Baseline,0.9500,0.9600,0.6900
2,Logistic Regression,1.0000,0.9400,0.8661
3,Random Forest (max_depth=6),1.0000,1.0000,0.9399


## 4. Errors and interpretation

*Where is the model wrong? What does it lean on? A short error analysis beats a big metric table.*

In [8]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
# 1. Feature Importance Analysis
importances = rf_model.feature_importances_
df_imp = pd.DataFrame({"Feature": feature_cols, "Importance": importances}).sort_values("Importance", ascending=False)

print("--- Feature Importances ---")
print(df_imp.to_string(index=False))

# 2. Inspect Concrete Error Cases
X_val_eval = X_val.copy()
X_val_eval['target_true'] = y_val
X_val_eval['rf_prob'] = rf_probs
X_val_eval['baseline_score'] = val_baseline_scores

# High probability but true label 0 (False Positive)
false_positives = X_val_eval[(X_val_eval['rf_prob'] > 0.8) & (X_val_eval['target_true'] == 0)].head(3)

# Low probability but true label 1 (False Negative)
false_negatives = X_val_eval[(X_val_eval['rf_prob'] < 0.2) & (X_val_eval['target_true'] == 1)].head(3)

print("\n--- Concrete False Positives (Model high, Actual low) ---")
print(false_positives)

print("\n--- Concrete False Negatives (Model low, Actual high) ---")
print(false_negatives)

--- Feature Importances ---
     Feature  Importance
 impressions    0.550818
         ctr    0.205867
      clicks    0.143859
    sessions    0.091270
avg_position    0.008185

--- Concrete False Positives (Model high, Actual low) ---
       impressions  clicks  avg_position  sessions       ctr  target_true  \
5780         521.0     4.0      3.291492       7.0  0.767754            0   
36910       1874.0    19.0     27.375307      43.0  1.013874            0   
3101        1292.0     9.0     10.368808      28.0  0.696594            0   

        rf_prob  baseline_score  
5780   0.887146            30.0  
36910  0.844775            70.0  
3101   0.924970           100.0  

--- Concrete False Negatives (Model low, Actual high) ---
       impressions  clicks  avg_position  sessions  ctr  target_true  \
53259         24.0     0.0     25.125000       1.0  0.0            1   
26965         31.0     0.0      5.935484       1.0  0.0            1   
61992         26.0     0.0      5.423077   

## 4. Errors and Interpretation

### 4.1 Feature Importances — What Does the Model Lean On?

Analyzing the fitted Random Forest tree splits reveals where the model places predictive weight across the March features:

| Feature | Importance Score | Domain Interpretation |
| :--- | :---: | :--- |
| **`impressions`** | **0.5508** (55.1%) | Dominant signal. Search volume is the primary filter because low-volume pages offer minimal upside even if optimized. |
| **`ctr`** | **0.2059** (20.6%) | Primary opportunity indicator. Highlights pages with strong visibility but low user engagement/click conversion. |
| **`clicks`** | **0.1439** (14.4%) | Direct traffic volume signal working alongside impressions to measure baseline engagement. |
| **`sessions`** | **0.0913** (9.1%) | Secondary cross-verification metric using GA4 web traffic. |
| **`avg_position`** | **0.0082** (0.8%) | Minimal individual weight, as positional impact is largely absorbed through the relationship between impressions and CTR. |

> **Sanity Check:** The model heavily prioritizes search visibility (`impressions`) and conversion efficiency (`ctr`), which aligns directly with business logic: a page must first have search demand before a content refresh can unlock meaningful traffic gains.

---

### 4.2 Error Analysis — Where Is the Model Wrong?

Inspecting concrete failure modes from the validation predictions provides direct insight into model limitations:

#### 1. False Positives ($P(\text{refresh}) > 0.84$, but Target = 0)
* **Index 36910:** `impressions` = 1,874, `ctr` = 1.01%, `avg_position` = 27.38 $\rightarrow$ **`rf_prob` = 0.8448**
  * *Why it failed:* The page sat barely above the 1.0% CTR boundary in March (1.01%) with high impressions and poor position. The model assigned a high refresh probability, but in April, subtle traffic shifts pushed it out of the refresh target definition.
* **Index 3101:** `impressions` = 1,292, `ctr` = 0.70%, `avg_position` = 10.37 $\rightarrow$ **`rf_prob` = 0.9250**
  * *Why it failed:* Hovering right around position 10 and 1,000 impressions makes this a classic threshold boundary edge case where minor month-over-month fluctuations flip the discrete outcome label.

#### 2. False Negatives ($P(\text{refresh}) < 0.14$, but Target = 1)
* **Index 53259:** `impressions` = 24, `ctr` = 0.0%, `avg_position` = 25.13 $\rightarrow$ **`rf_prob` = 0.1074**
* **Index 26965:** `impressions` = 31, `ctr` = 0.0%, `avg_position` = 5.94 $\rightarrow$ **`rf_prob` = 0.1318**
  * *Why it failed:* These pages had virtually zero search volume in March ($<35$ impressions), causing the model (which relies 55% on impression volume) to assign a near-zero refresh score. However, in April, these pages spiked in impressions while keeping a 0% CTR, triggering the Target = 1 label.
  * *Takeaway:* The model struggles to anticipate low-volume pages that suddenly gain search demand in subsequent months.

---

### 4.3 Methodological Critique: Signal Persistence vs. Independent Outcome

While validation metrics demonstrate strong ranking capabilities, a key architectural nuance exists in how the target is evaluated:

1. **Target Construction:** The target label is derived using similar impression and CTR thresholds in April as the input features in March.
2. **Persistence vs. Prediction:** Search Console signals are highly autocorrelated month-over-month. The model is largely capturing **signal persistence** (whether a page remains low-performing) rather than predicting an independent downstream business failure (such as a sudden organic session drop).
3. **Future Iteration:** Production targets should evolve from static threshold conditions to dynamic business outcome deltas, such as predicting a $\ge 20\%$ decline in organic sessions ($\Delta\text{Sessions} = \text{Sessions}_{t+1} - \text{Sessions}_t$).

## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.